# 01 · Synthetic multimodal data generation
Three **source datasets** are generated for 500 artificial patients across five classes
(CN, SMC, EMCI, LMCI, AD):

| Modality | Source format | Deliberate real-world issues |
|---|---|---|
| Vision | 125 `.dcm`, 125 `.nii`, 125 `.tiff`, 125 `.jpg` | 4 ID conventions, 4 resolutions, 3 bit depths, scanner-vendor intensity scaling, noise / low-contrast / flipped images |
| Text | clinical notes CSV | abbreviations, 6 duplicate notes, 8 orphan notes, 3 empty notes |
| Numeric | biomarker CSV (21 columns) | 15 duplicate rows, MCAR missingness, unit error at one site, decimal typos, label and sex coding variants |

A shared latent disease severity drives all modalities with independent noise, so each modality is
only partly informative and **fusion is needed** (SMC vs CN lives mostly in text, EMCI vs LMCI in
cognitive scores, LMCI vs AD in atrophy and function). All data are artificial.

In [1]:
import sys, json, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
from IPython.display import Image, display, Markdown
FIG = ROOT / 'reports' / 'figures'
def show(name, width=900): display(Image(filename=str(FIG / f'{name}.png'), width=width))

In [2]:
from ad_mm.data import generate_synthetic as g
from ad_mm.config import CFG
print(CFG['data'])
# g.main()   # uncomment to regenerate (scripts/run_pipeline.sh already ran it)

{'n_patients': 500, 'class_counts': {'CN': 130, 'SMC': 80, 'EMCI': 110, 'LMCI': 100, 'AD': 80}, 'image_formats': ['dcm', 'nii', 'tiff', 'jpg'], 'raw_dir': 'data/raw', 'interim_dir': 'data/interim', 'processed_dir': 'data/processed', 'yolo_dir': 'data/yolo'}


In [3]:
raw = ROOT / 'data' / 'raw'
man = pd.read_csv(raw / 'imaging' / 'imaging_manifest.csv')
notes = pd.read_csv(raw / 'clinical_notes' / 'clinical_notes.csv')
tab = pd.read_csv(raw / 'tabular' / 'clinical_biomarkers.csv')
print('imaging studies:', len(man), '| notes:', len(notes), '| tabular rows:', len(tab))
display(man.head()); display(notes.head(3)); display(tab.head())

imaging studies: 500 | notes: 514 | tabular rows: 515


,file,format,shape,site,scanner,injected_defect
0,tiff/pt_0001.tiff,tiff,"(160, 160)",SITE-C,Philips,none
1,tiff/pt_0002.tiff,tiff,"(160, 160)",SITE-D,GE MEDICAL SYSTEMS,none
2,dicom/PT-0003.dcm,dcm,"(128, 128)",SITE-E,Philips,none
3,nifti/sub-0004_T1w.nii,nii,"(64, 64, 24)",SITE-C,GE MEDICAL SYSTEMS,none
4,jpg/PATIENT0005.jpg,jpg,"(192, 192)",SITE-A,GE MEDICAL SYSTEMS,none


,patient_id,visit_date,author_role,note_text
0,PT0017,2025-11-10,geriatrician,73-year-old female seen for pt c/o misplacing ...
1,pt-0487,2025-03-13,NP,80-year-old male seen for memory lapses notice...
2,PT0397,2025-08-02,neurologist,73-year-old male seen for concern about memory...


,patient_id,site,age,sex,education_yrs,apoe4_copies,mmse,moca,cdr_sb,adas_cog13,...,ecog_self,ecog_partner,hippocampus_vol_mm3,entorhinal_thick_mm,ventricles_vol_mm3,ptau217_pg_ml,ab42_40_ratio,nfl_pg_ml,gfap_pg_ml,dx
0,0400,SITE-C,67.0,M,14.0,0,29.0,29.0,0.0,9.9,...,1.14,NaN,7.597,3.37,20569.0,0.182,0.1094,19.6,136.2,CN
1,0451,SITE-A,73.0,M,15.0,1,25.0,22.0,2.0,27.4,...,1.86,1.94,7018.000,3.50,50077.0,0.587,0.0745,23.0,222.9,LMCI
2,PT-0061,SITE-D,76.0,F,14.0,0,27.0,26.0,1.3,12.8,...,1.95,1.37,6721.000,3.29,42942.0,0.377,0.0783,23.2,224.8,EMCI
3,PT-0116,SITE-E,71.0,F,16.0,0,29.0,28.0,0.0,13.6,...,1.28,1.05,7336.000,4.08,18611.0,0.204,0.1004,5.4,72.3,CN
4,PT-0428,SITE-B,73.0,M,12.0,2,25.0,25.0,2.1,16.8,...,1.97,1.78,7389.000,3.40,23215.0,0.499,0.0739,13.8,NaN,LMCI


In [4]:
for folder in ['dicom', 'nifti', 'tiff', 'jpg']:
    files = sorted((raw / 'imaging' / folder).iterdir())
    print(f'{folder:6s} {len(files):4d} files  e.g. {files[0].name}')

dicom   125 files  e.g. PT-0003.dcm
nifti   125 files  e.g. sub-0004_T1w.nii
tiff    125 files  e.g. pt_0001.tiff
jpg     125 files  e.g. PATIENT0005.jpg
